# Module 14 — Decorators

You have been reading `@` since module 00: `@pytest.mark.your_turn` in every test
file, then `@property`, `@classmethod` and `@staticmethod` in module 11, and
`@dataclass` in module 12. This is where it gets explained — and it turns out to be
module 04 plus one idea.

The one idea: **a function is a value.** Everything else follows.

## 1. A function is a value

`def` binds a name, exactly as `=` does — module 04 said so to explain why there is
no overloading. Here is the other half of that fact.

In [ ]:
def celsius_to_fahrenheit(value):
    return value * 1.8 + 32


converter = celsius_to_fahrenheit  # no call: the function itself
print(converter(21.7))

converters = [celsius_to_fahrenheit, abs, round]  # in a list, like any other value
print([f(-2.5) for f in converters])


def apply_twice(function, value):  # as an argument -- module 04 did this with sorted(key=...)
    return function(function(value))


print(apply_twice(celsius_to_fahrenheit, 0))

And a function can **return** a function. That is the piece that makes a decorator
possible: the returned function still reaches the names from where it was made, which
is the closure from module 04.

In [ ]:
def scaler(factor):
    def scale(value):  # a closure: it captures `factor`
        return value * factor

    return scale  # returned, not called


double = scaler(2)
triple = scaler(3)

print(double(21), triple(21))
print(type(double).__name__, double.__name__)

## 2. `@` is an assignment

A decorator is a function that takes a function and gives back something to bind to
the name instead. The `@` is notation for exactly that:

```python
@loud
def add(a, b): ...
```

means

```python
def add(a, b): ...
add = loud(add)
```

Nothing more. Once you read it that way, every decorator in the course becomes
ordinary.

In [ ]:
def loud(function):
    def wrapper(*args, **kwargs):  # *args/**kwargs from module 04: accept anything
        print(f"   calling {function.__name__}")
        result = function(*args, **kwargs)
        print(f"   {function.__name__} returned {result}")
        return result  # returning this is easy to forget, and the function then gives None

    return wrapper


@loud
def add(a, b):
    return a + b


print(add(1, 2))

In [ ]:
def loud(function):
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)

    return wrapper


def add(a, b):
    return a + b


add = loud(add)  # the same thing, written out
print(add(1, 2))

Java has annotations, and they are **not** this. `@Override` is metadata: the compiler
checks the claim and then discards it — it is not even in the class file. Others do
survive compilation and are read later by a framework or by reflection, which is how
`@Deprecated` and `@Transactional` work.

A Python decorator is not metadata. It **runs**, when the `def` is executed, and
whatever it returns is bound to the name. It may replace the function — `@loud` does,
and `add` no longer refers to what you wrote — or it may hand the same function back.
`@app.route("/")` in module 21 and `@app.get("/readings")` in module 23 are the second
kind: they register the function somewhere and return it unchanged. Either way the
decorator has run, which is the thing an annotation does not do.

## 3. What the wrapper loses

The name bound to `add` is now `wrapper`. So the function's own name, docstring and
signature are gone, and everything that reads them — `help()`, a traceback, a
documentation generator, `inspect` — sees the wrapper instead.

In [ ]:
import functools


def loud(function):
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)

    return wrapper


@loud
def add(a, b):
    """Add two numbers."""
    return a + b


# What is add.__name__ now?
assert add.__name__ == ...
assert add.__doc__ is ...

In [ ]:
import inspect


def loud(function):
    @functools.wraps(
        function
    )  # copies __name__, __qualname__, __doc__, __module__, __annotations__
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)

    return wrapper


@loud
def add(a, b):
    """Add two numbers."""
    return a + b


print(add.__name__, "|", add.__doc__)
print(inspect.signature(add))
print(add.__wrapped__.__name__)  # wraps leaves a way back to the original

**`@functools.wraps(function)` on the wrapper, every time.** It is one line, it costs
nothing, and without it a stack trace names `wrapper` for every decorated function in
the program.

One thing it does not copy: the wrapper's own signature is still `(*args, **kwargs)`.
`inspect.signature(add)` shows `(a, b)` because it follows `__wrapped__`;
`inspect.signature(add, follow_wrapped=False)` shows what is really there.

## 4. A decorator that takes an argument

`@repeat(3)` is not a decorator; it is a **call** that returns one. That is why there
is a third level of nesting, and why the extra pair of brackets is not decoration:

```python
@repeat(3)
def ping(): ...
```

means

```python
def ping(): ...
ping = repeat(3)(ping)
```

In [ ]:
def repeat(times):  # takes the argument, returns the decorator
    def decorator(function):  # takes the function, returns the wrapper
        @functools.wraps(function)
        def wrapper(*args, **kwargs):
            return [function(*args, **kwargs) for _ in range(times)]

        return wrapper

    return decorator


@repeat(3)
def ping():
    return "p"


print(ping())
print(repeat(2)(ping)())  # written out, and now applied twice over

Three levels, one job each: the outer takes the decorator's arguments, the middle
takes the function, the inner takes the call's arguments. If you find yourself lost in
a decorator, count the levels — that is usually the whole confusion.

## 5. Stacking

Decorators stack, and the order matters. Predict this one.

In [ ]:
def outer(function):
    def wrapper():
        return "outer(" + function() + ")"

    return wrapper


def inner(function):
    def wrapper():
        return "inner(" + function() + ")"

    return wrapper


@outer
@inner
def base():
    return "base"


assert base() == ...

**Bottom up.** The stack is applied from the `def` outwards, so

```python
@outer
@inner
def base(): ...
```

is `base = outer(inner(base))`. The one nearest the `def` wraps first and therefore
ends up innermost; the one at the top runs first when the function is called.

That is why `@app.route` on top of an authentication decorator behaves differently
from the reverse, and why `@functools.cache` above the call counter in exercise 06
counts something different from `@functools.cache` below it.

## 6. Decorating a method

Nothing special is needed — `self` arrives as the first positional argument and
`*args` catches it.

In [ ]:
def trace(function):
    @functools.wraps(function)
    def wrapper(*args, **kwargs):
        return f"[{function.__name__}] {function(*args, **kwargs)}"

    return wrapper


class Sensor:
    def __init__(self, tag):
        self.tag = tag

    @trace
    def describe(self, unit):
        return f"{self.tag} in {unit}"


print(Sensor("TH-04").describe("C"))

Which also explains `@property` from module 11. It is a decorator whose return value
is not a function at all: it is a descriptor object, which intercepts attribute
access. `@classmethod` and `@staticmethod` are the same idea — the returned object
changes what happens when the name is looked up on an instance.

You will not write one of those. Reading them is enough: they are functions that were
handed your function and gave back something else.

## 7. The ones in the standard library

Three you will use rather than write.

In [ ]:
@functools.cache  # remembers every argument it has seen
def expensive(n):
    expensive.calls += 1
    return n * 2


expensive.calls = 0

expensive(1)
expensive(1)  # not called again
expensive(2)

print("real calls:", expensive.calls)
print(expensive.cache_info())

`@functools.cache` is right for a pure function that is called repeatedly with the
same arguments. It is wrong for anything that reads the outside world — a cached
`load_settings()` never notices the file changing — and it is wrong when the arguments
are unbounded, because the cache is never cleared and the memory only grows.
`@functools.lru_cache(maxsize=128)` is the version with a limit.

The other two worth knowing now: `@functools.wraps`, which you have already met, and
`@contextlib.contextmanager` from module 09 — which is a decorator that turns a
generator into a context manager, and now reads as exactly that.

## 8. When not to write one

A decorator moves behaviour away from the code it affects. That is the point, and it
is also the cost: the reader of `add(1, 2)` cannot see that anything happens before or
after, and finding out means going to look at `@loud`.

Worth it for concerns that are genuinely orthogonal and repeated — logging, timing,
caching, retrying, registering, access control. Not worth it for something that
happens once, or for anything the reader of the call site needs to know about. When
the decorator changes what the function *returns*, as `@repeat` does above, be
careful: the signature now lies, and nothing warns anybody.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

Module 15 is `pytest`, and from there the feedback in this course changes: instead of
an expected output to match, you write the tests.